In [ ]:
!mkdir -p data output
!curl -o data/cow.obj https://raw.githubusercontent.com/facebookresearch/pytorch3d/main/docs/tutorials/data/cow_mesh/cow.obj
!curl -o data/cow.mtl https://raw.githubusercontent.com/facebookresearch/pytorch3d/main/docs/tutorials/data/cow_mesh/cow.mtl
!curl -o data/cow_texture.png https://raw.githubusercontent.com/facebookresearch/pytorch3d/main/docs/tutorials/data/cow_mesh/cow_texture.png

In [ ]:
!pip install "git+https://github.com/facebookresearch/pytorch3d.git"
!pip install matplotlib tqdm imageio

In [ ]:
import os
import gc
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
from IPython.display import clear_output

from pytorch3d.io import load_objs_as_meshes, save_obj
from pytorch3d.structures import Meshes
from pytorch3d.utils import ico_sphere

from pytorch3d.renderer import (
    look_at_view_transform,
    FoVPerspectiveCameras,
    RasterizationSettings,
    MeshRenderer,
    MeshRasterizer,
    SoftSilhouetteShader,
    SoftPhongShader,
    BlendParams,
    PointLights,
    TexturesVertex
)

from pytorch3d.loss import (
    mesh_edge_loss,
    mesh_laplacian_smoothing,
    mesh_normal_consistency
)

# =========================================================
# 1. 环境初始化
# =========================================================

torch.cuda.empty_cache()
gc.collect()

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("🚀 Device:", device)

output_dir = "output_joint_dynamic"
os.makedirs(output_dir, exist_ok=True)

# =========================================================
# 2. 加载目标 Mesh
# =========================================================

obj_path = "./data/cow.obj"

if not os.path.exists(obj_path):
    raise FileNotFoundError(f"❌ Cannot find: {obj_path}")

target_mesh = load_objs_as_meshes([obj_path], device=device)

# normalize mesh
verts = target_mesh.verts_packed()

center = verts.mean(0)
scale = max((verts - center).abs().max(0)[0])

target_mesh.offset_verts_(-center)
target_mesh.scale_verts_(1.0 / float(scale))

# =========================================================
# 3. 多视角相机
# =========================================================

num_views = 24

elev = torch.linspace(-20, 20, num_views)
azim = torch.linspace(-180, 180, num_views)

R, T = look_at_view_transform(
    dist=2.7,
    elev=elev,
    azim=azim
)

cameras = FoVPerspectiveCameras(
    device=device,
    R=R,
    T=T
)

# =========================================================
# 4. Soft Rasterization
# =========================================================

blend_params = BlendParams(
    sigma=1e-4,
    gamma=1e-4,
    background_color=(1.0, 1.0, 1.0)
)

raster_settings_soft = RasterizationSettings(
    image_size=256,
    blur_radius=np.log(1. / 1e-4 - 1.) * blend_params.sigma,
    faces_per_pixel=50,
)

raster_settings_rgb = RasterizationSettings(
    image_size=256,
    blur_radius=0.0,
    faces_per_pixel=1,
)

# silhouette renderer
silhouette_renderer = MeshRenderer(
    rasterizer=MeshRasterizer(
        cameras=cameras,
        raster_settings=raster_settings_soft
    ),
    shader=SoftSilhouetteShader(
        blend_params=blend_params
    )
)

# 更柔和的灯光（防止纹理被高光抹掉）
lights = PointLights(
    device=device,
    location=[[0.0, 0.0, -3.0]],

    ambient_color=((0.85, 0.85, 0.85),),
    diffuse_color=((0.3, 0.3, 0.3),),
    specular_color=((0.0, 0.0, 0.0),)
)

# RGB renderer
rgb_renderer = MeshRenderer(
    rasterizer=MeshRasterizer(
        cameras=cameras,
        raster_settings=raster_settings_rgb
    ),
    shader=SoftPhongShader(
        device=device,
        cameras=cameras,
        lights=lights
    )
)

# =========================================================
# 5. 生成 GT
# =========================================================

target_mesh_extend = target_mesh.extend(num_views)

with torch.no_grad():

    target_silhouette = silhouette_renderer(
        target_mesh_extend
    )[..., 3]

    target_rgb = rgb_renderer(
        target_mesh_extend
    )[..., :3]

# =========================================================
# 6. 初始化 Sphere
# =========================================================

# subdivision=4 可以恢复更多细节
src_mesh = ico_sphere(4, device)

verts_shape = src_mesh.verts_packed().shape

# geometry deformation
deform_verts = torch.zeros(
    verts_shape,
    device=device,
    requires_grad=True
)

# vertex color texture
verts_rgb = torch.full(
    [1, verts_shape[0], 3],
    0.5,
    device=device,
    requires_grad=True
)

# =========================================================
# 7. Joint Optimization
# =========================================================

optimizer = torch.optim.Adam([
    {
        "params": deform_verts,
        "lr": 0.003
    },
    {
        "params": verts_rgb,
        "lr": 0.02
    }
])

epochs = 800

loss_history = []
sil_history = []
rgb_history = []

# =========================================================
# 8. Optimization Loop
# =========================================================

for i in tqdm(range(epochs), desc="Optimizing"):

    optimizer.zero_grad()

    # ---------------------------------
    # deform mesh
    # ---------------------------------

    new_mesh = src_mesh.offset_verts(deform_verts)

    new_mesh.textures = TexturesVertex(
        verts_features=verts_rgb
    )

    mesh_extend = new_mesh.extend(num_views)

    # ---------------------------------
    # render
    # ---------------------------------

    pred_silhouette = silhouette_renderer(
        mesh_extend
    )[..., 3]

    pred_rgb = rgb_renderer(
        mesh_extend
    )[..., :3]

    # ---------------------------------
    # dynamic rgb weight
    # ---------------------------------

    rgb_weight = min(1.0, i / 300)

    # ---------------------------------
    # losses
    # ---------------------------------

    loss_sil = F.mse_loss(
        pred_silhouette,
        target_silhouette
    )

    # L1 更适合 texture
    loss_rgb = F.l1_loss(
        pred_rgb,
        target_rgb
    )

    # regularization
    loss_lap = mesh_laplacian_smoothing(
        new_mesh,
        method="uniform"
    )

    loss_edge = mesh_edge_loss(
        new_mesh
    )

    loss_normal = mesh_normal_consistency(
        new_mesh
    )

    # deformation penalty
    loss_deform = torch.mean(
        deform_verts ** 2
    )

    # ---------------------------------
    # total loss
    # ---------------------------------

    loss = (
        1.0 * loss_sil
        + rgb_weight * loss_rgb
        + 0.3 * loss_lap
        + 1.0 * loss_edge
        + 0.01 * loss_normal
        + 0.05 * loss_deform
    )

    # ---------------------------------
    # backward
    # ---------------------------------

    loss.backward()

    optimizer.step()

    # clamp texture
    verts_rgb.data.clamp_(0.0, 1.0)

    # ---------------------------------
    # record
    # ---------------------------------

    loss_history.append(loss.item())
    sil_history.append(loss_sil.item())
    rgb_history.append(loss_rgb.item())

    # =====================================================
    # visualization
    # =====================================================

    if i % 50 == 0 or i == epochs - 1:

        clear_output(wait=True)

        print(f"\nEpoch {i}/{epochs}")
        print(f"Total Loss: {loss.item():.6f}")
        print(f"Silhouette Loss: {loss_sil.item():.6f}")
        print(f"RGB Loss: {loss_rgb.item():.6f}")
        print(f"RGB Weight: {rgb_weight:.3f}")

        # save obj
        current_verts = new_mesh.verts_packed().detach()
        current_faces = new_mesh.faces_packed().detach()

        save_obj(
            os.path.join(output_dir, f"mesh_epoch_{i:04d}.obj"),
            current_verts,
            current_faces
        )

        # visualization
        fig, ax = plt.subplots(1, 3, figsize=(15, 5))

        ax[0].imshow(
            target_silhouette[0].cpu().numpy(),
            cmap="gray"
        )
        ax[0].set_title("GT Silhouette")

        ax[1].imshow(
            target_rgb[0].cpu().numpy()
        )
        ax[1].set_title("GT RGB")

        ax[2].imshow(
            pred_rgb[0].detach().cpu().numpy()
        )
        ax[2].set_title(f"Prediction Epoch {i}")

        for a in ax:
            a.axis("off")

        plt.tight_layout()
        plt.show()

# =========================================================
# 9. 保存最终 Mesh
# =========================================================

final_mesh = src_mesh.offset_verts(
    deform_verts
)

final_verts = final_mesh.verts_packed().detach()
final_faces = final_mesh.faces_packed().detach()

save_obj(
    os.path.join(output_dir, "final_cow.obj"),
    final_verts,
    final_faces
)

# =========================================================
# 10. Loss Curve
# =========================================================

plt.figure(figsize=(10, 5))

plt.plot(
    loss_history,
    label="Total Loss",
    linewidth=2
)

plt.plot(
    sil_history,
    label="Silhouette Loss",
    linewidth=2
)

plt.plot(
    rgb_history,
    label="RGB Loss",
    linewidth=2
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title("Joint Shape + Texture Optimization")

plt.grid(True, linestyle="--", alpha=0.5)

plt.legend()

plt.savefig(
    os.path.join(output_dir, "loss_curve.png"),
    dpi=300
)

plt.show()

print("\n🎯 Optimization Finished!")

print(f"\n📁 Results saved to: {output_dir}")